In [3]:
!pip3 install tiktoken


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

AttributeError: module 'tiktoken' has no attribute 'get_decoding'

In [5]:
pip install datasets


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

In [9]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total characters:", len(raw_text))
print(raw_text[:999])

Total characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?

In [10]:
encoded_text = tokenizer.encode(raw_text)
print(len(encoded_text))

5145


In [11]:
encoded_sample = encoded_text[50:]

In [15]:
context_size = 4
x = encoded_sample[:context_size]
y = encoded_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:       {y}")

x: [290, 4920, 2241, 287]
y:       [4920, 2241, 287, 257]


In [21]:
for i in range(1, context_size+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    print(context, "--->", desired)

[290] ---> 4920
[290, 4920] ---> 2241
[290, 4920, 2241] ---> 287
[290, 4920, 2241, 287] ---> 257


In [23]:
for i in range(1, context_size+1):
    context = encoded_sample[:i]
    desired = encoded_sample[i]
    
    print(tokenizer.decode(context), "--->", tokenizer.decode([desired]))

 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a


In [24]:
pip install torch


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [36]:
import torch
from torch.utils.data import Dataset, DataLoader

class DatasetV1(Dataset):

    def __init__(self, text, tokenizer, context_length, stride):

        self.input_ids = []
        self.output_ids = []

        token_ids = tokenizer.encode(
            text,
            allowed_special={"<|endoftext|>"}
        )

        for i in range(0, len(token_ids) - context_length, stride):

            input_chunk = token_ids[i:i + context_length]

            output_chunk = token_ids[i + 1:i + context_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(output_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.output_ids[index]

In [37]:
def create_dataloader(
    txt,
    batch_size=4,
    context_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0
):

    tokenizer = tiktoken.get_encoding("gpt2")

    dataset = DatasetV1(
        txt,
        tokenizer,
        context_length,
        stride
    )

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [38]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [41]:
import torch 

dataloader = create_dataloader(
    raw_text,
    batch_size=1,
    context_length=4,
    stride=1,
    shuffle=False
)

data_iter = iter(dataloader)

data_iter = iter(dataloader)

for i in range(5):
    input_batch, target_batch = next(data_iter)

    print(f"Batch {i + 1}")
    print("Input :", input_batch)
    print("Target:", target_batch)

Batch 1
Input : tensor([[  40,  367, 2885, 1464]])
Target: tensor([[ 367, 2885, 1464, 1807]])
Batch 2
Input : tensor([[ 367, 2885, 1464, 1807]])
Target: tensor([[2885, 1464, 1807, 3619]])
Batch 3
Input : tensor([[2885, 1464, 1807, 3619]])
Target: tensor([[1464, 1807, 3619,  402]])
Batch 4
Input : tensor([[1464, 1807, 3619,  402]])
Target: tensor([[1807, 3619,  402,  271]])
Batch 5
Input : tensor([[1807, 3619,  402,  271]])
Target: tensor([[ 3619,   402,   271, 10899]])
